# SVG glyph fingerprints: vector SVG → canonical geometry → SDF → PCA

ZIP format:

```text
dataset.zip
├── tremble/*.svg
├── bass/*.svg
├── digit2/*.svg
└── digit3/*.svg
```

The goal is exploratory: check whether real variants of the same glyph become nearest neighbours after geometric normalization and PCA compression. No centerline/skeleton is used.


In [ ]:
#@title 1. Install
!pip -q install svgelements shapely scikit-learn matplotlib pandas


In [ ]:
#@title 2. Imports
from pathlib import Path
import math, zipfile, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shapely
from shapely.geometry import Polygon
from shapely.ops import unary_union
from shapely.affinity import affine_transform
from svgelements import SVG, Path as SvgPath, Shape, Move, Close
from sklearn.decomposition import PCA
from sklearn.metrics import pairwise_distances
from google.colab import files

WORK = Path("/content/glyph-pca")
DATASET_DIR = WORK / "dataset"
if WORK.exists():
    shutil.rmtree(WORK)
DATASET_DIR.mkdir(parents=True)
print("Shapely:", shapely.__version__)


In [ ]:
#@title 3. Upload ZIP
EXPECTED_CLASSES = {"tremble", "bass", "digit2", "digit3"}

uploaded = files.upload()
zip_names = [n for n in uploaded if n.lower().endswith(".zip")]
if len(zip_names) != 1:
    raise ValueError(f"Expected exactly one ZIP, got {zip_names}")

zip_path = WORK / zip_names[0]
zip_path.write_bytes(uploaded[zip_names[0]])

with zipfile.ZipFile(zip_path) as zf:
    zf.extractall(DATASET_DIR)

def find_root(base):
    candidates = [base] + [p for p in base.iterdir() if p.is_dir()]
    for p in candidates:
        names = {x.name.lower() for x in p.iterdir() if x.is_dir()}
        if EXPECTED_CLASSES.issubset(names):
            return p
    raise ValueError(f"Folders not found: {EXPECTED_CLASSES}")

ROOT = find_root(DATASET_DIR)

items = []
for class_dir in sorted(p for p in ROOT.iterdir() if p.is_dir()):
    label = class_dir.name.lower()
    if label not in EXPECTED_CLASSES:
        continue
    for svg in sorted(class_dir.glob("*.svg")):
        items.append({"label": label, "name": svg.name, "path": svg})

df_files = pd.DataFrame(items)
display(df_files[["label", "name"]])
display(df_files.groupby("label").size().rename("count").to_frame())


## Vector parser

Curves are sampled directly from SVG Bézier/Arc geometry into Shapely polygons; there is no rasterization here. Compound subpaths are combined with XOR, which is a practical fit for typical glyph outlines with holes.


In [ ]:
#@title 4. SVG → Shapely helpers
VECTOR_STEP = 0.08  #@param {type:"number"}

def sample_segment(seg, max_step):
    try:
        length = max(float(seg.length(error=1e-6)), 1e-12)
    except Exception:
        length = max(float(seg.length()), 1e-12)
    n = max(2, int(math.ceil(length / max_step)) + 1)
    return [(float(seg.point(float(t)).x), float(seg.point(float(t)).y))
            for t in np.linspace(0, 1, n)]

def path_to_rings(path, max_step):
    rings, current = [], []
    for seg in path:
        if isinstance(seg, Move):
            if len(current) >= 3:
                if current[0] != current[-1]: current.append(current[0])
                rings.append(current)
            current = [(float(seg.end.x), float(seg.end.y))]
            continue

        pts = sample_segment(seg, max_step)
        if not current and pts: current.append(pts[0])
        if pts: current.extend(pts[1:])

        if isinstance(seg, Close):
            if len(current) >= 3:
                if current[0] != current[-1]: current.append(current[0])
                rings.append(current)
            current = []

    if len(current) >= 3:
        if current[0] != current[-1]: current.append(current[0])
        rings.append(current)
    return rings

def load_svg_geometry(svg_path):
    svg = SVG.parse(str(svg_path), reify=True)
    rings = []

    for element in svg.elements():
        if isinstance(element, SvgPath):
            path = element
        elif isinstance(element, Shape):
            try:
                path = SvgPath(element)
                path.reify()
            except Exception:
                continue
        else:
            continue

        if not len(path) or str(getattr(element, "fill", "black")).lower() == "none":
            continue
        rings.extend(path_to_rings(path, VECTOR_STEP))

    if not rings:
        raise ValueError(f"No filled paths in {svg_path.name}")

    geometry = None
    for ring in rings:
        poly = Polygon(ring)
        if not poly.is_valid: poly = poly.buffer(0)
        if poly.is_empty: continue
        geometry = poly if geometry is None else geometry.symmetric_difference(poly)

    if geometry is None or geometry.is_empty:
        raise ValueError(f"No polygon geometry from {svg_path.name}")

    geometry = geometry.buffer(0)
    if geometry.geom_type == "GeometryCollection":
        geometry = unary_union([
            g for g in geometry.geoms
            if g.geom_type in ("Polygon", "MultiPolygon")
        ])
    return geometry

def iter_polygons(g):
    if g.geom_type == "Polygon": return [g]
    if g.geom_type == "MultiPolygon": return list(g.geoms)
    return [x for x in getattr(g, "geoms", []) if x.geom_type == "Polygon"]

def plot_geometry(ax, g, title=""):
    for p in iter_polygons(g):
        x, y = p.exterior.xy
        ax.plot(x, y)
        for hole in p.interiors:
            x, y = hole.xy
            ax.plot(x, y)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")
    ax.set_title(title)


In [ ]:
#@title 5. Parse all SVGs
records = []
for row in df_files.itertuples(index=False):
    try:
        g = load_svg_geometry(row.path)
        records.append({
            "label": row.label, "name": row.name,
            "path": row.path, "geometry": g,
            "area": g.area, "boundary_length": g.boundary.length
        })
    except Exception as e:
        print("FAILED:", row.path, e)

glyphs = pd.DataFrame(records)
display(glyphs[["label", "name", "area", "boundary_length"]])
print(f"Parsed {len(glyphs)} / {len(df_files)}")


In [ ]:
#@title 6. Originals
n = len(glyphs)
cols = min(5, n)
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(3.1*cols, 4*rows))
axes = np.array(axes, dtype=object).reshape(-1)
for ax, row in zip(axes, glyphs.itertuples(index=False)):
    plot_geometry(ax, row.geometry, f"{row.label}\n{row.name}")
for ax in axes[n:]: ax.axis("off")
plt.tight_layout()
plt.show()


## Canonical normalization

Modes:
- `scale_only`: translation + isotropic scale
- `pca_rotate`: translation + PCA orientation + isotropic scale
- `pca_whiten`: PCA orientation + independent scaling by eigenvalues, so aspect stretch is largely removed

PCA axis sign is stabilized using third moments of sampled boundary points.


In [ ]:
#@title 7. Canonical transform
NORMALIZATION = "pca_rotate"  #@param ["scale_only", "pca_rotate", "pca_whiten"]
BOUNDARY_SAMPLES = 512  #@param {type:"integer"}
TARGET_RADIUS = 0.8  #@param {type:"number"}

def boundary_samples(g, n=BOUNDARY_SAMPLES):
    b = g.boundary
    ds = np.linspace(0, b.length, n, endpoint=False)
    return np.array([(b.interpolate(float(d)).x, b.interpolate(float(d)).y) for d in ds])

def canonical_transform(g, mode):
    pts = boundary_samples(g)
    mean = pts.mean(axis=0)
    centered = pts - mean

    if mode == "scale_only":
        A = np.eye(2)
        evals = np.array([1.0, 1.0])
    else:
        cov = np.cov(centered.T, bias=True)
        evals, evecs = np.linalg.eigh(cov)
        order = np.argsort(evals)[::-1]
        evals = np.maximum(evals[order], 1e-12)
        evecs = evecs[:, order]
        A = evecs.T
        if mode == "pca_whiten":
            A = np.diag(1 / np.sqrt(evals)) @ A
        elif mode != "pca_rotate":
            raise ValueError(mode)

    q = centered @ A.T
    for axis in range(2):
        if np.mean(q[:, axis] ** 3) < 0:
            A[axis, :] *= -1

    q = centered @ A.T
    radius = max(np.max(np.linalg.norm(q, axis=1)), 1e-12)
    A *= TARGET_RADIUS / radius

    offset = -A @ mean
    g2 = affine_transform(
        g,
        [A[0,0], A[0,1], A[1,0], A[1,1], offset[0], offset[1]]
    )
    return g2, evals

normalized, eigs = [], []
for row in glyphs.itertuples(index=False):
    g2, ev = canonical_transform(row.geometry, NORMALIZATION)
    normalized.append(g2)
    eigs.append(ev)

glyphs["normalized"] = normalized
glyphs["eigenvalues"] = eigs
print("Mode:", NORMALIZATION)


In [ ]:
#@title 8. Normalized vector geometries
n = len(glyphs)
cols = min(5, n)
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(3.1*cols, 4*rows))
axes = np.array(axes, dtype=object).reshape(-1)
for ax, row in zip(axes, glyphs.itertuples(index=False)):
    plot_geometry(ax, row.normalized, f"{row.label}\n{row.name}")
    ax.set_xlim(-1, 1)
    ax.set_ylim(1, -1)
for ax in axes[n:]: ax.axis("off")
plt.suptitle(f"Canonical geometry: {NORMALIZATION}")
plt.tight_layout()
plt.show()


## Signed Distance Field (SDF)

A fixed grid removes dependence on SVG path count, segment count, path start point, and winding direction. Each cell stores signed distance to the glyph boundary; values are clipped so distant background does not dominate PCA.


In [ ]:
#@title 9. SDF descriptors
GRID_SIZE = 32  #@param {type:"integer"}
GRID_EXTENT = 1.0  #@param {type:"number"}
SDF_CLIP = 0.30  #@param {type:"number"}

axis = np.linspace(-GRID_EXTENT, GRID_EXTENT, GRID_SIZE)
xx, yy = np.meshgrid(axis, axis)
grid_points = shapely.points(xx.ravel(), yy.ravel())

def geometry_to_sdf(g):
    d = np.asarray(shapely.distance(g.boundary, grid_points), dtype=float)
    inside = np.asarray(shapely.contains(g, grid_points), dtype=bool)
    d[inside] *= -1
    d = np.clip(d, -SDF_CLIP, SDF_CLIP) / SDF_CLIP
    return d.reshape(GRID_SIZE, GRID_SIZE)

sdfs = [geometry_to_sdf(g) for g in glyphs["normalized"]]
glyphs["sdf"] = sdfs
X = np.asarray([x.ravel() for x in sdfs])
labels = glyphs["label"].to_numpy()
names = glyphs["name"].to_numpy()
short_names = [f"{a}/{Path(b).stem}" for a,b in zip(labels,names)]
print("Descriptor matrix:", X.shape)


In [ ]:
#@title 10. SDF thumbnails
n = len(glyphs)
cols = min(5, n)
rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(3*cols, 3.4*rows))
axes = np.array(axes, dtype=object).reshape(-1)
for ax, row in zip(axes, glyphs.itertuples(index=False)):
    ax.imshow(row.sdf)
    ax.set_title(f"{row.label}\n{row.name}")
    ax.axis("off")
for ax in axes[n:]: ax.axis("off")
plt.suptitle(f"SDF {GRID_SIZE}×{GRID_SIZE} / {NORMALIZATION}")
plt.tight_layout()
plt.show()


In [ ]:
#@title 11. Fit dataset-level PCA
REQUESTED_COMPONENTS = 8  #@param {type:"integer"}
N_COMPONENTS = min(REQUESTED_COMPONENTS, X.shape[0]-1, X.shape[1])
if N_COMPONENTS < 1:
    raise ValueError("Need at least two valid glyphs")

pca = PCA(n_components=N_COMPONENTS)
F = pca.fit_transform(X)
glyphs["fingerprint"] = list(F)

print("Fingerprint dimensions:", N_COMPONENTS)
print("Explained variance:", f"{pca.explained_variance_ratio_.sum():.1%}")

display(pd.DataFrame({
    "PC": np.arange(1, N_COMPONENTS+1),
    "variance": pca.explained_variance_ratio_,
    "cumulative": np.cumsum(pca.explained_variance_ratio_)
}))


In [ ]:
#@title 12. Explained variance
plt.figure(figsize=(8,4))
plt.plot(np.arange(1,N_COMPONENTS+1),
         np.cumsum(pca.explained_variance_ratio_), marker="o")
plt.xlabel("PCA components")
plt.ylabel("Cumulative explained variance")
plt.ylim(0,1.05)
plt.grid(True)
plt.show()


In [ ]:
#@title 13. PCA scatter
if N_COMPONENTS < 2:
    print("Need at least 2 PCs")
else:
    fig, ax = plt.subplots(figsize=(9,7))
    for label in sorted(set(labels)):
        m = labels == label
        ax.scatter(F[m,0], F[m,1], s=60, label=label)
        for i in np.where(m)[0]:
            ax.annotate(names[i], (F[i,0],F[i,1]), fontsize=8)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.legend()
    ax.grid(True)
    plt.show()


In [ ]:
#@title 14. Fingerprint distance matrix
DISTANCE_METRIC = "euclidean"  #@param ["euclidean", "cosine"]

D = pairwise_distances(F, metric=DISTANCE_METRIC)
np.fill_diagonal(D, np.nan)

fig, ax = plt.subplots(figsize=(max(8,len(glyphs)*0.55),
                                max(7,len(glyphs)*0.50)))
ax.imshow(D)
ax.set_xticks(np.arange(len(short_names)))
ax.set_yticks(np.arange(len(short_names)))
ax.set_xticklabels(short_names, rotation=90, fontsize=8)
ax.set_yticklabels(short_names, fontsize=8)
ax.set_title(f"PCA fingerprint distances: {N_COMPONENTS} PCs")
plt.tight_layout()
plt.show()


In [ ]:
#@title 15. Nearest neighbours
TOP_K = 4  #@param {type:"integer"}

def top1_accuracy(Dm):
    correct = 0
    for i in range(len(glyphs)):
        valid = np.where(~np.isnan(Dm[i]))[0]
        j = valid[np.argmin(Dm[i,valid])]
        correct += int(labels[j] == labels[i])
    return correct / len(glyphs)

rows = []
for i in range(len(glyphs)):
    valid = np.where(~np.isnan(D[i]))[0]
    order = valid[np.argsort(D[i,valid])][:TOP_K]
    r = {"query": short_names[i],
         "top1_same_class": labels[order[0]] == labels[i]}
    for rank,j in enumerate(order,1):
        r[f"nn{rank}"] = short_names[j]
        r[f"d{rank}"] = float(D[i,j])
    rows.append(r)

display(pd.DataFrame(rows))
print("Exploratory leave-self-out top-1:",
      f"{top1_accuracy(D):.1%}")
print("PCA is fitted on the whole tiny dataset: this is a geometry sanity check, not production accuracy.")


In [ ]:
#@title 16. Raw SDF vs PCA
D_raw = pairwise_distances(X, metric=DISTANCE_METRIC)
np.fill_diagonal(D_raw, np.nan)

display(pd.DataFrame([
    {"descriptor": f"raw SDF ({X.shape[1]} dims)",
     "top1_same_class": top1_accuracy(D_raw)},
    {"descriptor": f"PCA ({N_COMPONENTS} dims)",
     "top1_same_class": top1_accuracy(D)}
]))


## Automatic normalization comparison

This is the key experiment for the original hypothesis: do PCA rotation and whitening actually help, or do they remove useful class information?


In [ ]:
#@title 17. Compare normalization modes
MODES = ["scale_only", "pca_rotate", "pca_whiten"]
benchmark_rows = []
benchmark_data = {}

for mode in MODES:
    geoms = [canonical_transform(g, mode)[0] for g in glyphs["geometry"]]
    mode_X = np.asarray([geometry_to_sdf(g).ravel() for g in geoms])

    nc = min(REQUESTED_COMPONENTS, mode_X.shape[0]-1, mode_X.shape[1])
    model = PCA(n_components=nc)
    mode_F = model.fit_transform(mode_X)
    mode_D = pairwise_distances(mode_F, metric=DISTANCE_METRIC)
    np.fill_diagonal(mode_D, np.nan)

    benchmark_rows.append({
        "normalization": mode,
        "pca_dims": nc,
        "explained_variance": model.explained_variance_ratio_.sum(),
        "top1_same_class": top1_accuracy(mode_D)
    })
    benchmark_data[mode] = (geoms, mode_X, mode_F, model, mode_D)

benchmark = pd.DataFrame(benchmark_rows)
display(benchmark.sort_values("top1_same_class", ascending=False))


In [ ]:
#@title 18. Numeric fingerprints
fp = pd.DataFrame(F, columns=[f"pc{i+1}" for i in range(N_COMPONENTS)])
fp.insert(0, "glyph", short_names)
display(fp)


# What to look at

1. **Normalized geometries** — do same-class glyphs become visually closer?
2. **Nearest neighbours** — does each glyph find another glyph of the same class first?
3. **Normalization benchmark** — does `pca_rotate` or `pca_whiten` beat plain scale normalization?
4. **Raw SDF vs PCA** — does PCA compression preserve or improve neighbourhood structure?

If the result is promising, the next PoC should use more samples and proper train/validation separation. In production the learned PCA is only a mean vector + matrix multiply, so Python is not required for inference.
